In [1]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Carregamento dos dados de treino e teste
train = pd.read_csv("/kaggle/input/competitions/campeonato-inteli-modulo-3-2026/train.csv")
test = pd.read_csv("/kaggle/input/competitions/campeonato-inteli-modulo-3-2026/test.csv")

# Removendo colunas de cotações para evitar qualquer tipo de vazamento de dados
train_clean = train.drop(columns=['odd_casa', 'odd_empate', 'odd_visitante'])

# Mapeando os resultados para a ordem alfabética exigida: A=0, D=1, H=2
le = LabelEncoder()
y_train = le.fit_transform(train_clean['Res'])

def extrair_atributos(df):
    d = df.copy()

    # Transformação matemática da diferença de Elo em probabilidade esperada de vitória
    d['elo_prob_home'] = 1.0 / (1.0 + 10.0 ** (-d['elo_diff'] / 400.0))

    # Módulo de métricas relativas (diferença direta entre mandante e visitante)
    d['form_pts_diff'] = d['form_pts_home'] - d['form_pts_away']
    d['season_ppg_diff'] = d['season_ppg_home'] - d['season_ppg_away']
    d['form_gf_diff'] = d['form_gf_home'] - d['form_gf_away']
    d['form_ga_diff'] = d['form_ga_home'] - d['form_ga_away']
    d['host_visitor_diff'] = d['home_form_as_host'] - d['away_form_as_visitor']
    d['rest_days_diff'] = d['rest_days_home'] - d['rest_days_away']

    # Interação entre momento recente e probabilidade de Elo
    d['elo_x_form'] = d['elo_prob_home'] * d['form_pts_diff']

    colunas_finais = [
        'elo_prob_home',
        'elo_diff',
        'form_pts_diff',
        'season_ppg_diff',
        'form_gf_diff',
        'form_ga_diff',
        'host_visitor_diff',
        'rest_days_diff',
        'elo_x_form',
        'h2h_home_pts'
    ]
    return d[colunas_finais]

X_train = extrair_atributos(train_clean).fillna(0)
X_test = extrair_atributos(test).fillna(0)

# Padronização das variáveis
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Aplicação de pesos temporais para dar relevância progressiva às temporadas mais recentes
temporada_max = train_clean['Season'].max()
pesos_temporais = np.exp((train_clean['Season'] - temporada_max) / 5.0)

# Modelo 1: Regressão Logística altamente regularizada para manter estabilidade
modelo_logistico = LogisticRegression(C=0.008, max_iter=2000, random_state=42)
calibrador_logistico = CalibratedClassifierCV(modelo_logistico, method='sigmoid', cv=5)
calibrador_logistico.fit(X_train_scaled, y_train, sample_weight=pesos_temporais)

# Modelo 2: Árvore de decisão com controle rigoroso de profundidade
modelo_gb = HistGradientBoostingClassifier(
    learning_rate=0.015,
    max_depth=3,
    l2_regularization=4.0,
    max_iter=200,
    random_state=42
)
calibrador_gb = CalibratedClassifierCV(modelo_gb, method='sigmoid', cv=5)
calibrador_gb.fit(X_train_scaled, y_train, sample_weight=pesos_temporais)

# Combinação ponderada das previsões (70% modelo linear / 30% modelo de árvore)
preds_logistico = calibrador_logistico.predict_proba(X_test_scaled)
preds_gb = calibrador_gb.predict_proba(X_test_scaled)
preds_finais = (preds_logistico * 0.70) + (preds_gb * 0.30)

# Montagem do arquivo final mantendo a ordem estrita das colunas (A, D, H)
submissao = pd.DataFrame({
    'Id': test['Id'],
    'A': preds_finais[:, 0],
    'D': preds_finais[:, 1],
    'H': preds_finais[:, 2]
})

submissao.to_csv("ali-submissao4.csv", index=False)

print("Processamento concluído com sucesso. O arquivo ali-submissao4.csv está pronto.")

Processamento concluído com sucesso. O arquivo ali-submissao4.csv está pronto.
